# Notebook 0: 汇总各简单运动场景的最优参数集

扫描 `data/json/` 下所有贝叶斯优化结果，按运动类型分类并展示参数详情。

**说明**: 本 notebook 为独立参考文档，展示各运动场景的优化参数。
Notebook 03 直接从 JSON 文件加载参数，不再依赖本 notebook 的输出。

In [ ]:
import json
import re
from pathlib import Path

import numpy as np
import pandas as pd

PROJECT_ROOT = Path(r"D:\data\PPG_HeartRate\Algorithm\outline-PPGtoHR")
DATA_DIR = PROJECT_ROOT / "docs" / "research" / "data"
JSON_DIR = DATA_DIR / "json"

## 1. 运动场景定义与文件前缀映射

文件命名约定：`multi_{prefix}{N}.csv`，如 `multi_bobi1.csv`。
`PREFIX_MAP` 将中文拼音前缀映射为内部运动类别名。

In [2]:
EXERCISES = {
    "jump_rope": {"prefix": "tiaosheng", "label": "跳绳", "complexity": "simple"},
    "arm_curl": {"prefix": "wanju", "label": "手臂弯举", "complexity": "simple"},
    "push_up": {"prefix": "fuwo", "label": "俯卧撑", "complexity": "simple"},
    "jumping_jack": {"prefix": "kaihe", "label": "开合跳", "complexity": "simple"},
    "burpee": {"prefix": "bobi", "label": "波比跳", "complexity": "complex"},
}
# 反向映射: 拼音前缀 -> 运动类别名
PREFIX_MAP = {info["prefix"]: name for name, info in EXERCISES.items()}

print("运动场景:")
for name, info in EXERCISES.items():
    tag = "[简单]" if info["complexity"] == "simple" else "[复杂]"
    print(f"  {tag} {info['label']} ({name}) -> prefix={info['prefix']}")

运动场景:
  [简单] 跳绳 (jump_rope) -> prefix=tiaosheng
  [简单] 手臂弯举 (arm_curl) -> prefix=wanju
  [简单] 俯卧撑 (push_up) -> prefix=fuwo
  [简单] 开合跳 (jumping_jack) -> prefix=kaihe
  [复杂] 波比跳 (burpee) -> prefix=bobi


## 2. 扫描 JSON 优化结果

从 `data/json/` 读取所有 `*-best_params.json` 文件，按文件名提取运动类别和样本名。

In [ ]:
def parse_json_filename(fp):
    """从 JSON 文件名解析运动类别、样本 stem、ppg_mode、filter.

    文件名格式: multi_{prefix}{N}-{ppg_mode}-{filter}-best_params.json
    """
    pattern = r"^(multi_[a-z]+(?:\d+))-(\w+)-(\w+)-best_params\.json$"
    m = re.match(pattern, fp.name)
    if not m:
        return None
    stem = m.group(1)
    ppg_mode = m.group(2)
    filt = m.group(3)
    prefix_match = re.match(r"multi_([a-z]+)", stem)
    if not prefix_match:
        return None
    prefix = prefix_match.group(1)
    exercise = PREFIX_MAP.get(prefix)
    if exercise is None:
        return None
    return {
        "stem": stem,
        "exercise": exercise,
        "ppg_mode": ppg_mode,
        "adaptive_filter": filt,
    }


def scan_json_params(json_dir):
    """扫描 json_dir 下所有 best_params 文件并解析."""
    results = []
    for fp in sorted(json_dir.glob("*-best_params.json")):
        parsed = parse_json_filename(fp)
        if parsed is None:
            print(f"  跳过无法解析的文件: {fp.name}")
            continue
        with open(fp, "r", encoding="utf-8") as f:
            data = json.load(f)
        results.append({**parsed, "source": str(fp), **data})
    return results


all_results = scan_json_params(JSON_DIR)
print(f"共扫描到 {len(all_results)} 个优化结果:\n")
for r in all_results:
    label = EXERCISES[r["exercise"]]["label"]
    print(f"  {r['stem']}: {label}, min_err_hf={r.get('min_err_hf', float('nan')):.2f} BPM")

## 3. 优化效果对比

In [4]:
rows = []
for r in all_results:
    info = EXERCISES[r["exercise"]]
    rows.append({
        "exercise": info["label"],
        "type": info["complexity"],
        "stem": r.get("stem", "?"),
        "ppg_mode": r.get("ppg_mode", "green"),
        "filter": r.get("adaptive_filter", "lms"),
        "min_err_hf": r.get("min_err_hf", float("nan")),
        "min_err_acc": r.get("min_err_acc", float("nan")),
    })

df_summary = pd.DataFrame(rows)
if not df_summary.empty:
    df_summary = df_summary.sort_values(["type", "exercise", "stem"])
    display(df_summary)
else:
    print("未找到任何优化结果!")

,exercise,type,stem,ppg_mode,filter,min_err_hf,min_err_acc
0,波比跳,complex,multi_bobi1,green,lms,7.116816,11.065277
1,波比跳,complex,multi_bobi2,green,lms,7.456812,4.078431
2,俯卧撑,simple,multi_fuwo1,green,lms,3.364724,4.293569
3,开合跳,simple,multi_kaihe2,green,lms,3.364532,4.147234
5,手臂弯举,simple,multi_wanju1,green,lms,1.736733,1.730169
4,跳绳,simple,multi_tiaosheng2,green,lms,2.618239,2.616918


## 4. 各运动场景参数详情

展示各简单运动场景的最优参数 (来自 JSON `best_para_hf`)。
Notebook 03 使用 `load_report()` + `_merge()` 加载这些参数。

In [ ]:
from collections import defaultdict

results_by_exercise = defaultdict(list)
for r in all_results:
    results_by_exercise[r["exercise"]].append(r)

# 简单运动: 取 min_err_hf 最小的结果展示参数
print("简单运动最优参数 (best_para_hf):")
print("=" * 70)
for name in ["jump_rope", "arm_curl", "push_up", "jumping_jack"]:
    info = EXERCISES[name]
    results = results_by_exercise.get(name, [])
    if not results:
        print(f"  {info['label']} ({name}): 无优化结果")
        continue
    best = min(results, key=lambda r: r.get("min_err_hf", float("inf")))
    print(f"\n  {info['label']} ({name}) [from {best['stem']}]:")
    print(f"    min_err_hf = {best['min_err_hf']:.2f} BPM")
    for k, v in best["best_para_hf"].items():
        print(f"    {k}: {v}")

# 波比跳基线
print(f"\n{'=' * 70}")
print("波比跳基线:")
for r in results_by_exercise.get("burpee", []):
    print(f"  {r['stem']}: min_err_hf={r['min_err_hf']:.2f} BPM")

## 5. 参数就绪确认

确认各简单运动场景均有可用优化结果。

In [ ]:
missing = [EXERCISES[k]["label"] for k in EXERCISES
           if EXERCISES[k]["complexity"] == "simple" and k not in results_by_exercise]

if missing:
    print("=" * 60)
    print("警告: 以下简单运动场景缺少优化结果:")
    for m in missing:
        print(f"  - {m}")
    print("\n请先对这些场景运行贝叶斯优化, 然后重新运行本 notebook.")
    print("=" * 60)
else:
    print(f"所有 {len([k for k in EXERCISES if EXERCISES[k]['complexity'] == 'simple'])} 个简单运动场景的参数已就绪")
    print("Notebook 03 将直接从 JSON 文件加载这些参数 (使用 load_report + _merge)")